In [1]:
#| default_exp transform.transform

In [2]:
#| export
from __future__ import annotations

import logging

import numpy as np
import pandas as pd

logger = logging.getLogger("myproj.transform")

# Transformationen

## Setup

In [3]:
import numpy as np

from myproj.pipeline import run_import, run_pre_filter, run_cleaning, run_post_filter

pd.set_option("display.max_columns", 60)

logger = logging.getLogger("myproj.transform")

emdat_raw, sea_level_raw = run_import()
emdat, sea_level = run_pre_filter(emdat_raw, sea_level_raw)
emdat, sea_level = run_cleaning(emdat, sea_level)
emdat_flood = run_post_filter(emdat)

print(f"EMDAT nach Pipeline: {emdat.shape[0]:,} Zeilen, {emdat.shape[1]} Spalten")
print(f"Sea-Level nach Pipeline: {sea_level.shape[0]:,} Zeilen, {sea_level.shape[1]} Spalten")

2026-05-13 21:47:27 | myproj.pipeline      | INFO     | run_import | start
2026-05-13 21:47:27 | myproj.io            | INFO     | Lade Rohdatei: public_emdat_1991_2024.xlsx
2026-05-13 21:47:31 | myproj.io            | INFO     | load_raw_data | file: public_emdat_1991_2024.xlsx | rows: 20657 | cols: 47
2026-05-13 21:47:31 | myproj.io            | INFO     | Lade Rohdatei: omi_climate_sl_medsea_area_averaged_anomalies_19990220_P20250729.nc
2026-05-13 21:47:31 | myproj.io            | INFO     | load_raw_data | file: omi_climate_sl_medsea_area_averaged_anomalies_19990220_P20250729.nc | variables: 2 | dimensions: {'time': 9405}
2026-05-13 21:47:31 | myproj.pipeline      | INFO     | run_import | done
2026-05-13 21:47:31 | myproj.pipeline      | INFO     | run_pre_filter | start
2026-05-13 21:47:31 | myproj.transform     | INFO     | select_columns | cols: 47 → 18
2026-05-13 21:47:31 | myproj.transform     | INFO     | filter_to_sea_level_coverage | rows: 20657 → 16699 | removed before: 3

EMDAT nach Pipeline: 16,699 Zeilen, 18 Spalten
Sea-Level nach Pipeline: 9,405 Zeilen, 3 Spalten


## Datum konstruieren
EMDAT speichert das Datum als separate Float-Spalten (`Start Year`, `Start Month`, `Start Day`). Für den Sea-Level-Join werden datetime-Objekte benötigt.
Fehlende Monate werden mit 1 (Januar), fehlende Tage mit 1 aufgefüllt. Diese Approximation ist unvermeidlich und wird durch `start_date_quality` transparent dokumentiert. Beim Interpretieren von `sea_level_at_start` muss berücksichtigt werden, dass Ereignisse mit `month_and_day_imputed` ein ungenaueres Datum haben.

In [4]:
emdat_flood[["Start Year", "Start Month", "Start Day"]].dtypes

Start Year       int64
Start Month    float64
Start Day      float64
dtype: object

In [5]:
#| export
def _make_event_date(df: pd.DataFrame, prefix: str) -> pd.Series:
    return pd.to_datetime(
        pd.DataFrame({
            "year":  pd.to_numeric(df[f"{prefix} Year"],  errors="coerce"),
            "month": pd.to_numeric(df[f"{prefix} Month"], errors="coerce").fillna(1),
            "day":   pd.to_numeric(df[f"{prefix} Day"],   errors="coerce").fillna(1),
        }),
        errors="coerce",
    )

In [6]:
#| export
def _date_quality(df: pd.DataFrame, prefix: str) -> pd.Series:
    month_missing = df[f"{prefix} Month"].isna()
    day_missing   = df[f"{prefix} Day"].isna()
    return np.select(
        [month_missing & day_missing, month_missing, day_missing],
        ["month_and_day_imputed", "month_imputed", "day_imputed"],
        default="complete",
    )

In [7]:
#| export
def add_event_dates(df: pd.DataFrame) -> pd.DataFrame:
    """Konstruiert start_date / end_date aus Year/Month/Day-Spalten und dokumentiert die Qualität."""
    df = df.copy()
    df["start_date"]         = _make_event_date(df, "Start")
    df["end_date"]           = _make_event_date(df, "End")
    df["start_date_quality"] = _date_quality(df, "Start")
    df["end_date_quality"]   = _date_quality(df, "End")
    n_complete = int((df["start_date_quality"] == "complete").sum())
    n_day      = int((df["start_date_quality"] == "day_imputed").sum())
    n_both     = int((df["start_date_quality"] == "month_and_day_imputed").sum())
    logger.info(
        "add_event_dates | rows: %d | quality: complete=%d, day_imputed=%d, month_and_day_imputed=%d",
        len(df), n_complete, n_day, n_both,
    )
    return df

In [8]:
emdat_flood = add_event_dates(emdat_flood)
print(emdat_flood[["start_date", "start_date_quality", "end_date", "end_date_quality"]].head())

2026-05-13 21:47:31 | myproj.transform     | INFO     | add_event_dates | rows: 292 | quality: complete=290, day_imputed=2, month_and_day_imputed=0


     start_date start_date_quality   end_date end_date_quality
4198 1999-11-12           complete 1999-11-15         complete
5036 2000-10-31           complete 2000-10-31         complete
5099 2023-12-30           complete 2024-01-03         complete
5191 2022-12-11           complete 2022-12-11         complete
5192 2022-12-11           complete 2022-12-22         complete


## Abgeleitete Spalten

Ergänzung um analyserelevante Spalten (Feature-Erweiterung).

- **`season`**: Saisonzuordnung aus dem Startmonat.
- **`event_duration_days`**: Dauer in Tagen aus `end_date − start_date + 1`.
- **`has_coordinates`**: Bool-Flag aus Latitude/Longitude. Für spätere geospatiale Auswertungen.

In [9]:
#| export
def _season_from_month(month: float) -> str:
    if pd.isna(month):
        return "unknown"
    m = int(month)
    if m in [12, 1, 2]:
        return "winter"
    if m in [3, 4, 5]:
        return "spring"
    if m in [6, 7, 8]:
        return "summer"
    return "autumn"

In [10]:
#| export
def add_derived_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Fügt season, event_duration_days und has_coordinates hinzu."""
    df = df.copy()
    df["season"]              = df["Start Month"].apply(_season_from_month)
    df["event_duration_days"] = ((df["end_date"] - df["start_date"]).dt.days + 1).clip(lower=1)
    df["has_coordinates"]     = df[["Latitude", "Longitude"]].notna().all(axis=1)
    logger.info(
        "add_derived_columns | added: season, event_duration_days, has_coordinates | rows: %d",
        len(df),
    )
    return df

In [11]:
emdat_flood = add_derived_columns(emdat_flood)
print(emdat_flood["season"].value_counts())

2026-05-13 21:47:31 | myproj.transform     | INFO     | add_derived_columns | added: season, event_duration_days, has_coordinates | rows: 292


season
autumn    117
winter     67
summer     56
spring     52
Name: count, dtype: int64


## Sea Level Lags

Sea-Level-Werte der `n_lags` Tage vor `start_date`. Erhöhter Meeresspiegel in den Vortagen kann Vorbedingungen wie Grundwasserspiegel oder Bodenfeuchte beeinflusst haben. Es werden Lags von 1-5 Tagen als Spalten eingebaut.

In [12]:
#| export
def add_sea_level_lags(
    df: pd.DataFrame,
    sea_level_ts: pd.DataFrame,
    n_lags: int = 5,
    sea_value: str = "MSL_filtered_GIA_corrected_adjusted",
) -> pd.DataFrame:
    """Fügt Sea-Level-Werte der n_lags Tage vor start_date als Lag-Spalten hinzu."""
    df = df.copy()
    sea_msl_idx = sea_level_ts.set_index("time")[sea_value]
    for n in range(1, n_lags + 1):
        df[f"sea_level_lag_{n}d"] = (df["start_date"] - pd.Timedelta(days=n)).map(sea_msl_idx)
    logger.info("add_sea_level_lags | n_lags: %d | rows: %d", n_lags, len(df))
    return df

>`add_sea_level_lags`wird erst in **`05_Join.ipynb`** vorgezeigt, da `sea_level_ts` bzw. der verknüpfte DataFrame benötigt wird.

## Sea Level Skalierung
`sea_level_at_start` liegt typischerweise zwischen −3 und +8 cm. Für die Analyse ist nicht der absolute Wert entscheidend, sondern wie anomal der Meeresspiegel relativ zur Messperiode war.

Der Z-Score (z = (x − μ) / σ) ist eine lineare Transformation. Relative Unterschiede und Reihenfolge bleiben identisch, die Daten werden nicht verzerrt. Als Referenzverteilung dient die gesamte Sea-Level-Zeitreihe (nicht nur Flood-Events). Rohwerte in cm bleiben als eigene Spalten erhalten.

| Neue Spalte | Beschreibung |
|---|---|
| `sea_level_at_start_z` | Z-Score von `sea_level_at_start` relativ zur Gesamtzeitreihe |
| `mean_sea_level_while_disaster_z` | Z-Score von `mean_sea_level_while_disaster` |

In [ ]:
#| export
def add_sea_level_z_scores(
    df: pd.DataFrame,
    sea_level_ts: pd.DataFrame,
    sea_value: str = "MSL_filtered_GIA_corrected_adjusted",
    sea_trend: str = "trend_MSL_filtered_GIA_corrected_adjusted",
) -> pd.DataFrame:
    """
    Fügt Z-Score-normierte Sea-Level-Spalten hinzu.
    Referenz: gesamte Sea-Level-Zeitreihe.
 
    Loggt zusätzlich die Detrend-Referenz: Mittel/Streuung der Residuen (MSL − Trend)
    über die volle Reihe.
    """
    df = df.copy()
    msl_mean = sea_level_ts[sea_value].mean()
    msl_std  = sea_level_ts[sea_value].std(ddof=0)
    df["sea_level_at_start_z"]            = (df["sea_level_at_start"] - msl_mean) / msl_std
    df["mean_sea_level_while_disaster_z"] = (df["mean_sea_level_while_disaster"] - msl_mean) / msl_std
 
    detrend_resid = sea_level_ts[sea_value] - sea_level_ts[sea_trend]
    resid_mean = detrend_resid.mean()
    resid_std  = detrend_resid.std(ddof=0)
    logger.info(
        "add_sea_level_z_scores | msl_ref: μ=%.4f σ=%.4f cm | detrend_ref: residual_μ=%.4f σ=%.4f cm | rows: %d",
        msl_mean, msl_std, resid_mean, resid_std, len(df),
    )
    return df

>`add_sea_level_z_scores`wird erst in **`05_Join.ipynb`** vorgezeigt, da `sea_level_ts` bzw. der verknüpfte DataFrame benötigt wird.